2 constitutions: 
- convolutional layer
- pooling layer

In [ ]:
import torch
from torch import nn

class Reshape(torch.nn.Module):
    def forward(self, x):
        return x.view(-1, 1, 28, 28)
        # batch size remain unchanged
        # 1 channel, 28 height, 28 width

net = torch.nn.Sequential(
    Reshape(),
    nn.Conv2d(1, 6, kernel_size=5, padding=2),  # 1 input channel, 6 output channels, 5x5 kernel, padding of 2
    nn.Sigmoid(),
    nn.AvgPool2d(2, stride=2), # 2x2 pooling, stride of 2
    nn.Conv2d(6, 16, kernel_size=5), nn.Flatten(), # flatten to pass into linear multilayer perceptron
    nn.Linear(16 * 5 * 5, 120), nn.Sigmoid(), 
    nn.Linear(120, 84), nn.Sigmoid(),
    nn.Linear(84, 10)
)


In [ ]:
X = torch.rand(size=(1, 1, 28, 28), dtype=torch.float32)
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__, 'output shape:\t', X.shape)

In [ ]:
def evaluate_accuracy_gpu(net, data_iter, device=None):
    if isinstance(net, torch.nn.Module):
        net.eval() # set the model to evaluation mode
        if not device:
            device = next(iter(net.parameters())).device
            # if device is not specified, look at where the model parameters are and use that device
    metric = Accumulator(2) # num of correct predictions, num of predictions
    for X, y in data_iter:
        if isinstance(X, list):
            # required for BERT fine-tuning (to be covered later)
            X = [x.to(device) for x in X]
        else:
            X = X.to(device)
        y = y.to(device)
        metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]

def Accumulator(n):
    """Sum a list of numbers over time."""
    return [0.0] * n

def accuracy(y_hat, y):
    """Compute the number of correct predictions."""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())